# 🧠 EXACT 2026 - Track 1: Logic-Based Educational QA
### Hướng dẫn chạy thử nghiệm Pipeline trên Google Colab với Google Drive Cache

Notebook này hướng dẫn bạn thiết lập môi trường và chạy thử nghiệm hệ thống Neuro-Symbolic QA (Track 1) trên Google Colab sử dụng GPU. 

**⚠️ LƯU Ý QUAN TRỌNG:**
Do kiến trúc mới của hệ thống đã chuyển sang mô hình kết nối qua **Inference Server (OpenAI-compatible)** để đáp ứng kiểm tra của Ban tổ chức, bạn **PHẢI khởi chạy server chạy ngầm trước** rồi mới thực hiện chạy pipeline. Script chạy sẽ tự động kết nối qua API ở cổng 8000.


**✨ TÍNH NĂNG MỚI:**
Hệ thống đã hỗ trợ trích xuất đầy đủ và chính xác trường `premises_used` (chiếm 50% điểm số câu hỏi Type 1) từ cả bộ giải ký hiệu (Logic Tree) lẫn bộ giải neural fallback (LLM Chain-of-Thought). Kết quả offline sẽ được tự động chuyển đổi thành chỉ mục 1-based để khớp với nhãn ground truth.

---

## 1. Kết nối Google Drive và kiểm tra GPU

Chạy cell dưới đây để kết nối với Google Drive của bạn (nhằm truy cập thư mục chứa file precompiled wheel và model `Colab_Cache`). Đồng thời kiểm tra thông tin GPU.

*(Lưu ý: Đi tới **Runtime** -> **Change runtime type** -> chọn **T4 GPU** trước khi chạy)*

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra GPU
!nvidia-smi

## 2. Clone Repository và chuyển sang nhánh `test/track1`

In [ ]:
# Clone repo từ Github
!git clone https://github.com/AIVIETNAM-AIO-Triet-Descartes/EXACT2026-NeuroSymbolic-QA.git

# Chuyển con trỏ dòng lệnh vào thư mục dự án
%cd /content/EXACT2026-NeuroSymbolic-QA

# Chuyển sang nhánh test/track1
!git checkout test/track1

## 3. Cài đặt các thư viện phụ thuộc (Dependencies)

Chúng ta sẽ cài đặt các thư viện trong `requirements.txt`. Riêng đối với `llama-cpp-python`, ta sẽ cài đặt trực tiếp từ file `.whl` đã được biên dịch sẵn trong thư mục `Colab_Cache` trên Google Drive.

In [ ]:
# Đảm bảo đứng đúng thư mục dự án và cài đặt dependencies
%cd /content/EXACT2026-NeuroSymbolic-QA

!pip install -r requirements.txt

# Cài đặt llama-cpp-python từ file wheel (.whl) lưu trên Google Drive để tiết kiệm thời gian
!pip install /content/drive/MyDrive/Colab_Cache/llama_cpp_python-0.3.23-py3-none-linux_x86_64.whl

## 4. Chuẩn bị Mô hình Qwen 2.5 7B GGUF

Sao chép các file mô hình GGUF từ Drive vào ổ đĩa của Colab để tăng tốc độ load và suy luận.

In [ ]:
# Di chuyển vào thư mục dự án
%cd /content/EXACT2026-NeuroSymbolic-QA

# Sao chép các file GGUF từ Drive vào thư mục hiện tại
!cp /content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf .
!cp /content/drive/MyDrive/Colab_Cache/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf .

# Kiểm tra file đã tồn tại ở local hay chưa
import os
if os.path.exists("./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf"):
    print("✅ Copy model sang bộ nhớ Colab thành công!")
else:
    print("❌ Copy model thất bại hoặc chưa hoàn thành.")

## 5. Khởi động Inference Server chạy ngầm

Chọn một trong hai cách khởi động dưới đây để bật Server chạy ngầm trên cổng `8000`:

### **Cách 1: Khởi động Server qua `llama-cpp-python` (Khuyên dùng - Dùng trực tiếp file GGUF sẵn có)**

In [ ]:
%cd /content/EXACT2026-NeuroSymbolic-QA

# Khởi động llama-cpp-python server chạy ngầm ở cổng 8000
!nohup python3 -m llama_cpp.server --model ./qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf --port 8000 --n_gpu_layers -1 --n_ctx 2048 --alias Qwen/Qwen2.5-7B-Instruct > llama_server.log 2>&1 &

print("⏳ Đang khởi động Server... Vui lòng đợi khoảng 30 giây.")
import time
time.sleep(30)

# Kiểm tra trạng thái hoạt động của Server
!curl http://localhost:8000/v1/models

### **Cách 2: Khởi động Server bằng `vLLM` (Tự tải safetensors gốc từ HuggingFace)**
Cách này sẽ tải mô hình chính thức định dạng safetensors trực tiếp từ HuggingFace về bộ nhớ Colab, giúp tăng tốc độ suy luận tối đa.

In [ ]:
# Cài đặt vLLM
!pip install vllm

# Khởi động vLLM server chạy ngầm ở cổng 8000
# Lưu ý: GPU T4 có 15GB VRAM, cần giới hạn tối đa chiều dài token để tránh Out Of Memory
!nohup vllm serve Qwen/Qwen2.5-7B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.9 --max-model-len 2048 > vllm.log 2>&1 &

print("⏳ Đang khởi động vLLM Server và tải model từ HF... Quá trình này mất khoảng 2-3 phút.")
import time
time.sleep(120)

# Kiểm tra trạng thái hoạt động của Server
!curl http://localhost:8000/v1/models

## 6. Chạy thử nghiệm Pipeline (Track 1)

Sử dụng script `scripts/run_track1.py` để chạy pipeline. Hệ thống sẽ tự động kết nối qua REST API đang phục vụ ở cổng 8000 để chạy suy luận.

### 6.1 Chạy thử nhanh với 5 mẫu đầu tiên

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_test.json \
    --max-samples 5 \
    --evaluate

### 6.2 Chạy đánh giá trên dải dữ liệu cụ thể (ví dụ: Mẫu 50 đến 100)

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_50_100.json \
    --start-sample 50 \
    --end-sample 100 \
    --evaluate

### 6.3 Chạy toàn bộ Dataset (411 mẫu / ~808 câu hỏi)

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_full.json \
    --evaluate

## 7. Xem Kết Quả Đầu Ra
Sau khi chạy xong, các kết quả dự đoán và đánh giá chi tiết sẽ được lưu tại thư mục `output/`.
**Lưu ý:** Hãy kiểm tra trường `idx` trong tệp JSON kết quả. Trường này chứa danh sách các tiền đề được sử dụng (`premises_used`) dạng 1-based cho từng câu hỏi con.

In [ ]:
# Di chuyển vào thư mục dự án trước khi chạy
%cd /content/EXACT2026-NeuroSymbolic-QA

# Hiển thị 30 dòng đầu của file dự đoán để kiểm tra cấu trúc đầu ra
!head -n 30 output/predictions_test.json